This part of the pipeline processes the results from the Ancestral State Reconstruction using the Count tool, and annotates the exchanged genes using COG and BRITE.

### Paths and parameters

#### Pipeline input folders

In [ ]:
apa.file = "11-ASR-analysis/families.tsv"
tree.file = "11-ASR-analysis/input_ready.tree"
indices.path = "02-GTDB/subgroups"
ann.file = "11-ASR-analysis/matrix_annotations.tsv"
cog.file = "06-pangenome-annotation/mapper/all.emapper.annotations"

#### Pipeline output folders

In [ ]:
task_root = "11-ASR-analysis"
ASR_output = paste(task_root, "output", sep = "/")

system(paste('mkdir -p', task_root, ASR_output), intern = TRUE)

#### Tool pointers and parameters

In [ ]:
cog.cats.file = "utils/COG_cats.tsv"
ko.cats.json = "utils/kegg.json"
ko.cats.file = "utils/kegg.tsv"
kegg.parser = "utils/parse_kegg_tree.py"

KEGG term tree obtained by running
`curl https://rest.kegg.jp/get/br:ko00001/json > kegg.json`

In [ ]:
# For storing and wrangling data
library(data.table)
library(dplyr)
library(tidyr)
library(Matrix)

# For general plotting
library(ggplot2)
library(ggh4x)
library(IRdisplay)

# For colour palettes
library(RColorBrewer)
library(pals)

# For plotting phylogenies
library(ape)
library(ggtree)
library(ggimage)

# Miscellaneous
library(stringr)
library(forcats)

In [ ]:
getPalette = colorRampPalette(rev(stepped()))

## Reading files

#### Ancestral P/A data

In [ ]:
apa = fread(apa.file, data.table = FALSE)

In [ ]:
head(apa)

Genes that are gained in the root node (with label 1) are the LCA genes.

In [ ]:
common = apa$'1' != 0

#### Phylogeny

In [ ]:
tree = read.tree(tree.file)

In [ ]:
ggtree(tree) + geom_text(aes(label = label), nudge_x = -0.01)

#### Genome indices

In [ ]:
indices.files = list.files(indices.path)
groups = indices.files[grep('\\.list$', indices.files, invert = TRUE)]
groups

In [ ]:
taxa.sets = sapply(groups, function(x) read.table(paste(indices.path, x, sep = '/'))$V1)

In [ ]:
mrcas = lapply(taxa.sets, function(x) getMRCA(tree, tip = x))

In [ ]:
mrcas

#### annotations

In [ ]:
cog.ann = fread(ann.file, header = FALSE, col.names = c('family', 'annotation', 'category'), data.table = FALSE)

In [ ]:
head(cog.ann)

In [ ]:
cog.cats = read.table(cog.cats.file, sep = "\t", row.names = 1)

In [ ]:
cog.cats

In [ ]:
ko.ann = fread(cog.file, header = TRUE, skip = 4, select = c('#query', 'KEGG_ko'), data.table = FALSE, fill = TRUE)
ko.ann = ko.ann[-((nrow(ko.ann) - 3 + 1):nrow(ko.ann)),]
colnames(ko.ann) = c('family', 'KEGG_ko')
ko.ann = ko.ann %>% mutate(KEGG_ko = strsplit(str_replace(str_remove_all(KEGG_ko, 'ko:'), '-', 'Unannotated'), ",")) %>% unnest(KEGG_ko)

In [ ]:
head(ko.ann)

In [ ]:
system(paste('python', kegg.parser, ko.cats.json, ko.cats.file), intern = TRUE)
ko.cats = fread(ko.cats.file, header = TRUE, data.table = FALSE)
ko.cats = subset(ko.cats, !(A %in% c("Brite Hierarchies", "Organismal Systems", "Human Diseases")))

In [ ]:
head(ko.cats)

## Identifying the exchanged genes per node

### Auxiliary functions

The conversion between node IDs (number assigned internally by `ape`) and node labels (number assigned externally by `Count`) has been pushed in such a way by the conversion script for the inputs for `Count` (`convert_ASR_inputs.R`) that they can be easily interconverted by adding or subtracting the number of terminal leafs (i.e. the leafs that had a non-numerical label before input conversion).

In [ ]:
nodeID_to_nodeLabel = function(nodeID, tree) {
    return(nodeID - length(tree$tip.label))}
nodeLabel_to_nodeID = function(nodeLabel, tree) {
    return(nodeLabel + length(tree$tip.label))}

In [ ]:
## Returns the node ID of a given label in a given phylogeny object.
##
## PARAMS
## label      node label to get the ID of
## tree       phylogeny in the form of an ape tree object
##
## OUTPUT
## node ID of the tree node with the given label
##
findID = function(label, tree) {
    lab = str_replace_all(label, ' ', '_')
    ID.tip = which(tree$tip.label == lab)
    if (length(ID.tip) == 0) {
        ID.node = which(tree$node.label == lab)
        ID = nodeLabel_to_nodeID(ID.node, tree)
    }
    else {
        ID = ID.tip
    }
    return(ID)
}

In [ ]:
## Sorts the gene exchange events into gene gain and loss events, neglecting the number of genes exchanged.
## Also returns the full list of tree nodes and gene families.
##
## PARAMS
## apa      the gene presence table of the ASR produced by Count (presences.tsv)
## tree     the phylogeny in the form of an ape tree object
##
## OUTPUT
## a list of the binary gained and lost sparse matrices, and the array of tree nodes and gene families
##
identify_type = function(apa, tree) {
    families = apa$name
    nodes = colnames(apa)[str_detect(colnames(apa), "GCF [0-9]+\\.[0-9]|[1-9][0-9]+|[2-9]")]

    # Initialise two sparse matrices to represent whether a gene gain or loss, respectively, has taken place in a certain node
    gained.c = Matrix(FALSE, length(families), length(nodes)+1, sparse = TRUE)
    lost.c = gained.c

    # We'll compare the presence of gene families of each node with the one of its ancestor node
    for (node in nodes) {
        # Find the node label of the ancestor node using the IDs of the current node and its ancestor in the phylogeny object
        node.ID = findID(node, tree)
        ancestor.ID = tree$edge[tree$edge[,2] == node.ID, 1]
        ancestor = nodeID_to_nodeLabel(ancestor.ID, tree)

        # Get the presence of gene families in both tree nodes from the ASR presence table from Count
        pa.comp = subset(apa, select = c(ancestor, node))

        # Gained genes are present in this node but not in the ancestor node
        gained = pa.comp[,1] == 0 & pa.comp[,2] > 0
        gained.c[,node.ID] = gained

        # Lost genes are present in the ancestor node but not in this node
        lost = pa.comp[,1] > 0 & pa.comp[,2] == 0
        lost.c[,node.ID] = lost
    }
    
    res = list('gained' = gained.c, 'lost' = lost.c, 'nodes' = nodes, 'families' = families)
    return(res)
}

### Determining the exchange type of all gene families

In [ ]:
identified.c = identify_type(apa, tree)
gained.c = identified.c$gained
lost.c = identified.c$lost
nodes = identified.c$nodes
families = identified.c$families

In [ ]:
rm(apa)

In [ ]:
save(gained.c, lost.c, nodes, families, file = paste(ASR_output, 'sorted_genes.RData', sep = '/'))

### Overview trees

In [ ]:
## Returns a ape tree object supplemented with the number of gene exchange events by tree node
##
## PARAMS
## tree       phylogeny in the form an ape tree object
## ggl_data   sparse binary matrix indicating whether a certain type of gene exchange event has taken place in a certain tree node;
##            produced by identify_type()
##
## OUTPUT
## an ape tree object joined with a gene exchange metadata column
##
extend_tree_with_ggl = function(tree, ggl_data) {
    dt = data.frame(node = 1:(length(tree$node.label)+length(tree$tip.label)), trait = colSums(ggl_data))
    gt = full_join(fortify(tree), dt, by = "node")
    return(gt)
}

In [ ]:
gt.gained = extend_tree_with_ggl(tree, gained.c)
gt.gained

In [ ]:
dir.create(paste(ASR_output, 'gained', sep = '/'), recursive = TRUE)
svg(paste(ASR_output, 'gained', 'overview_tree.svg', sep = '/'), width = 8, height = 12)
ggtree(gt.gained, aes(color=.data$trait), size=0.5) +
    labs(colour='Total genes gained') +
    scale_color_gradientn(colours = magma(12), transform = "log10", na.value = "black")
dev.off()
display_svg(file = paste(ASR_output, 'gained', 'overview_tree.svg', sep = '/'))

In [ ]:
gt.lost = extend_tree_with_ggl(tree, lost.c)
gt.lost

In [ ]:
dir.create(paste(ASR_output, 'lost', sep = '/'), recursive = TRUE)
svg(paste(ASR_output, 'lost', 'overview_tree.svg', sep = '/'), width = 8, height = 12)
ggtree(gt.lost, aes(color=.data$trait), size=0.5) +
    labs(colour='Total genes lost') +
    scale_colour_gradientn(colours = magma(12), transform = "log10", na.value = "black")
dev.off()
display_svg(file = paste(ASR_output, 'lost', 'overview_tree.svg', sep = '/'))

## Characterising the exchanged genes of some nodes in particular

### Auxiliary functions

In [ ]:
## Gathers the gene family labels that were exchanged in a certain tree node
##
## PARAMS
## var.c      sparse binary matrix produced by identify_type() indicating in which tree node a certain gene family was exchanged
## families   array of gene family labels produced by identify_type()
## mrca       tree node to get the exchange gene family labels for
##
## OUTPUT
## a single-column dataframe listing the gene families that were exchanged in the given tree node
##
gather_genes = function(var.c, families, mrca) {
    var.mrca = as.data.frame(families[var.c[,mrca]])
    colnames(var.mrca) = c('family')
    return(var.mrca)
}

In [ ]:
## Aggregates and counts the COG annotations of an array of gene families
##
## PARAMS
## var        a single-column dataframe with a list of gene families; produced by gather_genes()
## full_cog   the COG category annotations listed by gene family
##
## OUTPUT
## a dataframe with relative COG category frequencies
##
aggregate_and_count_cog = function(var, full_cog) {
    cog = left_join(var, full_cog, by = 'family') # left join to preserve the unknown gene families
    cog.freq = cog %>% arrange(family) %>% count(category)

    # Redistribute the plural annotations (e.g. 'BE')
    for (c in cog.freq$category) {
        # Case for known gene families
        if (!is.na(c)) {
            # Only do something for plural annotations
            if (nchar(c) > 1) {
                idx = which(cog.freq$category == c)
                c.split = str_split(c, '')[[1]]
                # Redistributing plural annotations to the separate single annotation categories, creating a new category if non-existent
                for (cs in c.split) {
                    where = which(cog.freq$category == cs)
                    if (!length(where) == 0) {
                        cog.freq[where,"n"] = cog.freq[where,"n"] + cog.freq[idx,"n"]
                    }
                    else {
                        cog.freq[nrow(cog.freq)+1,] = list(cs, cog.freq[idx,"n"])
                    }
                }
                cog.freq = cog.freq[-c(idx),]
            }
        }
        # Unknown gene families end up with a NA label, so redistributing those to the unannotated category ('-')
        else {
            idx = which(is.na(cog.freq$category))
            cog.freq[which(cog.freq$category == '-'),'n'] = cog.freq[which(cog.freq$category == '-'),'n'] + cog.freq[idx,'n']
            cog.freq = cog.freq[-c(idx),]
        }
    }
    # Convert to relative frequencies
    cog.freq = arrange(cog.freq, category) %>% mutate(freq = n/sum(n)*100) %>% arrange(desc(n))
    return(cog.freq)
}

In [ ]:
## Gathers and counts both COG and KOG annotations of both gene exchange events for a certain tree node
##
## PARAMS
## gained.c      sparse binary matrix indicating whether a gene gain event has taken place in a tree node; produced by identify_type()
## lost.c        sparse binary matrix indicating whether a gene loss event has taken place in a tree node; produced by identify_type()
## families      array of gene family labels; produced by identify_type()
## mrca          tree node for which the annotations of the exchanged genes need to be examined
## cog           full COG annotation table for this genome set; expects the converted COG annotation list that was the input for Count
## kog           full KOG annotation table for this genome set; expects the processed KOG-BRITE table produced by notebook 07b2
## write_file    flag indicating whether the frequency tables should be written away (default = FALSE)
## output        output directory in which the frequency tables will be saved as tsv files; ignored if write_file is FALSE
## prefix        prefix for the filename of the frequency tables to distinguish different genome sets; ignored if write_file is FALSE
##
## OUTPUT
## a nested list of both gained and lost gene families with their COG and KOG annotations at all levels
##
gather_and_count = function(gained.c, lost.c, families, mrca, cog, write_file = FALSE, output = NULL, prefix = NULL) {
    ## Gained genes
    gained = gather_genes(gained.c, families, mrca)
    writeLines(c("Number of genes gained", nrow(gained)))
    cog.gained.freq = aggregate_and_count_cog(gained, cog)

    # Saving results
    if (write_file) {
        dir.create(paste(output, 'gained', sep = "/"), recursive = TRUE)
        path = paste(output, 'gained', prefix, sep = "/")
        write.table(cog.gained.freq, paste(path, 'cog', sep = "."), sep = "\t", quote = FALSE, col.names = TRUE, row.names = FALSE)
        writeLines('Results for gained genes saved!')
    }

    ## Lost genes
    lost = gather_genes(lost.c, families, mrca)
    writeLines(c("Number of genes lost", nrow(lost)))
    cog.lost.freq = aggregate_and_count_cog(lost, cog)

    # Saving results
    if (write_file) {
        dir.create(paste(output, 'lost', sep = "/"), recursive = TRUE)
        path = paste(output, 'lost', prefix, sep = "/")
        write.table(cog.lost.freq, paste(path, 'cog', sep = "."), sep = "\t", quote = FALSE, col.names = TRUE, row.names = FALSE)
        writeLines('Results for lost genes saved!')
    }

    res = list('gained' = 
               list('listed' = gained,
                    'cog' = cog.gained.freq),
               'lost' =
               list('listed' = lost,
                    'cog' = cog.lost.freq)
               )
    return(res)
}

### Selecting nodes to analyse

In [ ]:
asr.nodes = c(mrcas, 1024, 1216, 1684, 1561)

In [ ]:
asr.nodes = setNames(asr.nodes, c('all', 'Clostridiales', 'Lachnospirales', 'Oscillospirales', 'Peptostreptococcales', 
                                  'up1_Oscillospirales', 'subphylo_Lachnospirales', 'subphylo_Clostridiales', 'up2_Peptostreptococcales'))

### Processing the exchanged genes

In [ ]:
counted = lapply(asr.nodes, function(x) gather_and_count(gained.c, lost.c, families, x, cog.ann, TRUE, ASR_output, x))

#### Gained genes

In [ ]:
gained = lapply(counted, function(x) x$gained$listed %>% left_join(cog.ann, by = "family") 
                %>% left_join(ko.ann, by = 'family') %>% left_join(ko.cats, by = join_by(KEGG_ko == D)) 
                %>% arrange(annotation))

In [ ]:
lapply(gained, function(x) length(unique(x$family)))

In [ ]:
B.gained.freq = lapply(gained, function(x) x %>%
                       distinct(family, B) %>%
                       count(B) %>%
                       arrange(desc(n)) %>%
                       mutate_at('B', ~replace_na(., 'Unannotated'))
                       )

In [ ]:
B.gained.freq

In [ ]:
sapply(seq_along(B.gained.freq),
       function(x) write.table(B.gained.freq[[x]], paste(ASR_output, 'gained', 
                                                         paste(names(B.gained.freq)[[x]], 'kegg.B', sep = '.'),
                                                         sep = "/"),
                               sep = "\t", quote = FALSE, col.names = TRUE, row.names = FALSE),
       USE.NAMES = TRUE)

#### Lost genes

In [ ]:
lost = lapply(counted, function(x) x$lost$listed %>% left_join(cog.ann, by = "family") 
              %>% left_join(ko.ann, by = 'family') %>% left_join(ko.cats, by = join_by(KEGG_ko == D), relationship = 'many-to-many')
              %>% arrange(annotation))

In [ ]:
lapply(lost, function(x) length(unique(x$family)))

In [ ]:
B.lost.freq = lapply(lost, function(x) x %>% 
                     distinct(family, B) %>% 
                     count(B) %>% 
                     arrange(desc(n))
                     %>% mutate_at('B', ~replace_na(.,'Unannotated'))
                    )

In [ ]:
B.lost.freq

In [ ]:
sapply(seq_along(B.lost.freq),
       function(x) write.table(B.lost.freq[[x]], paste(ASR_output, 'lost', 
                                                       paste(names(B.lost.freq)[[x]], 'kegg.B', sep = '.'),
                                                       sep = "/"),
                               sep = "\t", quote = FALSE, col.names = TRUE, row.names = FALSE),
       USE.NAMES = TRUE)

## Plotting

#### Auxiliary lumping function and colour palette

In [ ]:
lump = function(count.df.0, top.n = 5, threshold = 3) {
    count.df = count.df.0 %>% filter(B != 'Unannotated')
    not.n.first = count.df[-c(1:top.n),]$B
    below.threshold = filter(count.df, n < threshold)$B
    other = c(not.n.first, below.threshold)
    res = count.df %>% count(Category = fct_collapse(count.df$B, Other = other), wt = n) 
    return(res)
}

#### Gained genes

In [ ]:
B.gained.freq.lumped = lapply(B.gained.freq, lump)

In [ ]:
B.gained.freq.lumped

In [ ]:
B.gained.freq.pivot = setNames(B.gained.freq.lumped, asr.nodes) %>% 
bind_rows(.id = 'node') %>% 
pivot_wider(names_from = Category, values_from = n) %>% 
select(node, Other, sort(names(.)))

old.cols = colnames(B.gained.freq.pivot)[-1]

B.gained.freq.pivot = B.gained.freq.pivot %>%
rename_with(~paste0(letters[seq_along(.)], '_', .)) %>%
rename('node' = 'a_node')

new.cols = colnames(B.gained.freq.pivot)[-1]

In [ ]:
colours = getPalette(length(new.cols))
colours[1] = '#f2f2f2'

In [ ]:
B.gained.freq.pivot

In [ ]:
pies.gained = nodepie(B.gained.freq.pivot, cols = new.cols)
pies.gained = lapply(pies.gained, function(g) g + scale_fill_manual(breaks = new.cols, values = colours))

In [ ]:
svg(paste(ASR_output, 'gained', 'overview_with_profile.svg', sep = "/"), height = 12, width = 8)
ggtree(gt.gained, aes(color=.data$trait), size=0.5) +
    labs(colour='Total genes gained') +
    scale_color_gradientn(colours = rev(magma(12)), transform = "log10", na.value = "black") +
    geom_inset(pies.gained, width = 0.1, height = 0.1, x = 'branch', vjust = -20)
dev.off()
display_svg(file = paste(ASR_output, 'gained', 'overview_with_profile.svg', sep = "/"))

#### Dummy plot to extract the legend from in Inkscape

In [ ]:
dummy_df = data.frame(Category = factor(new.cols), n = as.integer(1))

svg(paste(ASR_output, 'gained', 'dummy_profile.svg', sep = "/"), height = 6, width = 6)
ggplot(dummy_df, aes(x = '', y = n, fill = Category)) + 
    geom_col() + 
    coord_polar(theta = "y") + 
    scale_fill_manual(values = colours, labels = old.cols)
dev.off()
display_svg(file = paste(ASR_output, 'gained', 'dummy_profile.svg', sep = "/"))

#### Lost genes

In [ ]:
B.lost.freq.lumped = lapply(B.lost.freq, lump)

In [ ]:
B.lost.freq.lumped

In [ ]:
B.lost.freq.pivot = setNames(B.lost.freq.lumped, asr.nodes) %>% 
bind_rows(.id = 'node') %>% 
pivot_wider(names_from = Category, values_from = n) %>% 
select(node, Other, sort(names(.)))

old.cols = colnames(B.lost.freq.pivot)[-1]

B.lost.freq.pivot = B.lost.freq.pivot %>%
rename_with(~paste0(letters[seq_along(.)], '_', .)) %>%
rename('node' = 'a_node')

new.cols = colnames(B.lost.freq.pivot)[-1]

In [ ]:
colours = getPalette(length(new.cols))
colours[1] = '#F2F2F2'

In [ ]:
B.lost.freq.pivot

In [ ]:
pies.lost = nodepie(B.lost.freq.pivot, cols = new.cols)
pies.lost = lapply(pies.lost, function(g) g + scale_fill_manual(breaks = new.cols, values = colours))

In [ ]:
svg(paste(ASR_output, 'lost', 'overview_with_profile.svg', sep = "/"), height = 12, width = 8)
ggtree(gt.lost, aes(color=.data$trait), size=0.5) +
    labs(colour='Total genes lost') +
    scale_color_gradientn(colours = rev(magma(12)), transform = "log10", na.value = "black") +
    geom_inset(pies.lost, width = 0.1, height = 0.1, x = 'branch', vjust = -20)
dev.off()
display_svg(file = paste(ASR_output, 'lost', 'overview_with_profile.svg', sep = "/"))

#### Dummy plot to extract the legend from in Inkscape

In [ ]:
dummy_df = data.frame(Category = factor(new.cols), n = as.integer(1))

svg(paste(ASR_output, 'lost', 'dummy_profile.svg', sep = "/"), height = 6, width = 6)
ggplot(dummy_df, aes(x = '', y = n, fill = Category)) + 
    geom_col() + 
    coord_polar(theta = "y") + 
    scale_fill_manual(values = colours, labels = old.cols)
dev.off()
display_svg(file = paste(ASR_output, 'lost', 'dummy_profile.svg', sep = "/"))

## Characterising the genes in the LCA

In [ ]:
lca.genes = as.data.frame(families[common])
colnames(lca.genes) = c("family")

In [ ]:
nrow(lca.genes)

In [ ]:
lca.genes %>% left_join(ko.ann, by = 'family') %>% left_join(ko.cats, by = join_by(KEGG_ko == D)) %>% distinct(family, B, .keep_all = TRUE)

In [ ]:
lca.genes %>% left_join(ko.ann, by = 'family') %>% left_join(ko.cats, by = join_by(KEGG_ko == D)) %>% distinct(family, B, .keep_all = TRUE) %>% count(B) %>% arrange(desc(n))

In [ ]:
lca.cog.freq = aggregate_and_count_cog(lca.genes, cog.ann)

In [ ]:
dir.create(paste(ASR_output, 'common', sep = "/"), recursive = TRUE)
write.table(lca.cog.freq, paste(ASR_output, 'common', 'lca.cog', sep = "/"), 
            sep = "\t", quote = FALSE, col.names = TRUE, row.names = FALSE)

#### Plotting

In [ ]:
lca.cog.toplot = bind_rows(list('LCA' = lca.cog.freq), .id = 'cluster')
n.colors = length(unique(lca.cog.toplot$category))
present.cogs = sort(intersect(unique(lca.cog.toplot$category), rownames(cog.cats)))

svg(paste(ASR_output, 'common', 'COG_Freqs.svg', sep = "/"), height = 4, width = 5)
ggplot(lca.cog.toplot, aes(x = cluster, y = freq, fill = category)) +
  geom_bar(stat = "identity", position="stack") +
  scale_fill_manual(values = getPalette(n.colors), 
                    guide = guide_legend(ncol = 1, keyheight = 0.8, keywidth = 0.4),
                    labels = factor(paste0("(", present.cogs, ") ", cog.cats[present.cogs,]))) +
  scale_y_continuous(expand = c(0,0)) +
  scale_x_discrete(expand = c(0,0)) +
  labs(x = "", y = "% COGs", fill = "COG category") +
  theme(panel.spacing = unit(1, "lines"))
dev.off()
display_svg(file = paste(ASR_output, 'common', 'COG_Freqs.svg', sep = '/'))

## Saving and session info

In [ ]:
save.image(file = paste(ASR_output, 'env_output.RData', sep = "/"))

In [2]:
sessionInfo()

R version 4.3.2 (2023-10-31)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 22.04.4 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Brussels
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] ggtree_3.10.0      ape_5.7-1          pals_1.8           RColorBrewer_1.1-3
 [5] IRdisplay_1.1      ggh4x_0.2.8        ggplot2_3.5.0      Matrix_1.6-3      
